<a href="https://colab.research.google.com/github/orutkina/-./blob/main/tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Предсказание категории страйкбольного снаряжения

## Модуль 1. Создание модели

In [2]:
import pandas as pd

train = pd.DataFrame({
    'text': [
        'I love this product',
        'This is terrible',
        'Amazing experience',
        'Worst service ever',
        'Very happy with purchase'
    ],
    'sentiment': [
        'positive',
        'negative',
        'positive',
        'negative',
        'positive'
    ]
})

In [3]:
import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocessing_text(text):
    # только буквы
    text = re.sub('[^a-zA-Z]', ' ', text)

    # нижний регистр
    text = text.lower()

    # токены
    words = text.split()

    # удаление стоп-слов + лемматизация
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return ' '.join(words)

train['preprocess_text'] = train['text'].apply(preprocessing_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [4]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

encoder = LabelEncoder()

y = encoder.fit_transform(train['sentiment'])
y = to_categorical(y)

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=500)

X = vectorizer.fit_transform(train['preprocess_text']).toarray()

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras.layers import Dense

# делим данные
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# масштабируем
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# модель
model = keras.models.Sequential()
model.add(Dense(128, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(2, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.2500 - loss: 0.8110 - val_accuracy: 0.0000e+00 - val_loss: 0.8519
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 761ms/step - accuracy: 0.2500 - loss: 0.7759 - val_accuracy: 0.0000e+00 - val_loss: 0.8627
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 501ms/step - accuracy: 0.2500 - loss: 0.7421 - val_accuracy: 0.0000e+00 - val_loss: 0.8713
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step - accuracy: 0.2500 - loss: 0.7096 - val_accuracy: 0.0000e+00 - val_loss: 0.8785
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step - accuracy: 0.5000 - loss: 0.6785 - val_accuracy: 0.0000e+00 - val_loss: 0.8846
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - accuracy: 0.5000 - loss: 0.6485 - val_accuracy: 0.0000e+00 - val_loss: 0.8893
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 385ms/step - accuracy: 0.7500 - loss: 0.6196 - val_accuracy: 0.0000e+00 - val_loss: 0.8933
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step - accuracy: 0.7500 - loss: 0.5920 - val_acc

In [7]:
y_pred = model.predict(X_test)
y_pred = y_pred.argmax(axis=1)

# обратно в текст
y_pred = encoder.inverse_transform(y_pred)

y_pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step


array(['positive'], dtype=object)

In [10]:
test = pd.DataFrame({
    'text': [
        'I hate this',
        'This is awesome'
    ]
})

test['preprocess_text'] = test['text'].apply(preprocessing_text)

X_test_new = vectorizer.transform(test['preprocess_text']).toarray()
X_test_new = scaler.transform(X_test_new)

pred = model.predict(X_test_new).argmax(axis=1)
pred = encoder.inverse_transform(pred)

test['sentiment'] = pred
test

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step


,text,preprocess_text,sentiment
0,I hate this,hate,positive
1,This is awesome,awesome,positive


In [11]:
import pandas as pd

df_ads = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'text': [
        'продам страйкбольный автомат',
        'шлем защитный новый',
        'магазин для m4',
        'тактический жилет'
    ],
    'CategoryId': [None, None, None, None],
    'processed': [0, 0, 0, 0]
})

df_ads

,id,text,CategoryId,processed
0,1,продам страйкбольный автомат,None,0
1,2,шлем защитный новый,None,0
2,3,магазин для m4,None,0
3,4,тактический жилет,None,0


In [12]:
model
vectorizer

CountVectorizer(max_features=500)

In [15]:
def process_ads(df, model, vectorizer):
    mask = df['processed'] == 0

    for idx in df[mask].index:
        text = df.loc[idx, 'text']

        text_vec = vectorizer.transform([text]).toarray()

        # предсказание (исправлено!)
        pred = model.predict(text_vec)
        pred = pred.argmax(axis=1)[0]

        df.loc[idx, 'CategoryId'] = pred
        df.loc[idx, 'processed'] = 1

    return df

In [16]:
df_ads = process_ads(df_ads, model, vectorizer)

df_ads

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step


,id,text,CategoryId,processed
0,1,продам страйкбольный автомат,1,1
1,2,шлем защитный новый,1,1
2,3,магазин для m4,1,1
3,4,тактический жилет,1,1


In [17]:
code = """
import pandas as pd
import pickle
from tensorflow.keras.models import load_model

def load_artifacts():
    model = load_model('model.h5')

    with open('vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)

    with open('encoder.pkl', 'rb') as f:
        encoder = pickle.load(f)

    return model, vectorizer, encoder


def process_ads(df):
    model, vectorizer, encoder = load_artifacts()

    mask = df['processed'] == 0

    for idx in df[mask].index:
        text = df.loc[idx, 'text']

        text_vec = vectorizer.transform([text]).toarray()

        pred = model.predict(text_vec)
        pred = pred.argmax(axis=1)[0]
        pred = encoder.inverse_transform([pred])[0]

        df.loc[idx, 'CategoryId'] = pred
        df.loc[idx, 'processed'] = 1

    return df
"""

with open("process_ads.py", "w") as f:
    f.write(code)

In [18]:
import os
os.listdir()

['.config', 'process_ads.py', 'sample_data']

In [19]:
from process_ads import process_ads

In [20]:
from google.colab import files
files.download('process_ads.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Форма для отправки архивов - https://forms.gle/D9qUSJjNn8ZcSYDi6

для тестового набора данных с помощью вашей модели предскажите значение CategoryName, метрика для проверки f1-score(macro)

соревнование для отправки ответов на тестовую выборку -https://www.kaggle.com/t/cb73308e24f44addbefd8ba0c070380f